# Step 3 on Colab: train J4 (cube4) and evaluate native 4x4 vs embedded 3x3 / 2x2

Runtime -> Change runtime type -> **GPU**. Then run all cells.

* Checkpoints and results live on Google Drive (`DRIVE_ROOT`), so a disconnect loses at most one update cycle.
* **Re-running after a disconnect:** training resumes from the latest checkpoint and stops once `MAX_ITRS` total
  iterations are reached (if already reached, it exits immediately). Evaluation skips sections already in the JSON,
  unless the checkpoint changed since they were computed.
* `MAX_TIME_SEC` limits one training session (Colab sessions end); `MAX_ITRS` is the total training budget.
* If the repository is private, set `REPO_URL` to `https://<token>@github.com/bobcoelau12/Ruba.git`
  (a GitHub token with read access); do not save the notebook with the token in it.

In [ ]:
# ---- configuration ----
REPO_URL = "https://github.com/bobcoelau12/Ruba.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/Ruba_step3"
REPO_DIR = "/content/Ruba"

# training budget (short: this is a pipeline check, not the real training run)
MAX_ITRS = 20000            # total training iterations (resumable)
MAX_TIME_SEC = 3 * 3600     # per-session wall-clock limit
BACK_MAX = 10               # scramble depth for training states
BATCH_SIZE = 1000
STATES_PER_UPDATE = 200000  # training states generated per update cycle; a checkpoint is written after each cycle
NUM_TEST = 1000

# evaluation
EVAL_ARGS = "--depths 1 2 3 4 5 --baseline_depths 1 2 3 --num_per_depth 20 --weight 0.8 --max_nodes 200000"

SAVE_DIR = f"{DRIVE_ROOT}/saved_models"
MODEL_DIR = f"{SAVE_DIR}/cube4/current"
EVAL_JSON = f"{DRIVE_ROOT}/results/step3_eval.json"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
import os, subprocess
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# DeepCubeA only uses the GPU when CUDA_VISIBLE_DEVICES is set; otherwise it silently runs on CPU.
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONPATH"] = REPO_DIR
sys.path.insert(0, REPO_DIR)

import torch
from utils import nnet_utils
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
device, devices, on_gpu = nnet_utils.get_device()
print("device:", device, "devices:", devices, "on_gpu:", on_gpu, "-", torch.cuda.get_device_name(0))
assert on_gpu and device.type == "cuda", "DeepCubeA would fall back to CPU"

In [ ]:
# Train (resumes from MODEL_DIR if a checkpoint exists). Log: {SAVE_DIR}/cube4/output.txt
import pickle, os
itr_file = f"{MODEL_DIR}/train_itr.pkl"
print("checkpoint itr before:", pickle.load(open(itr_file, "rb")) if os.path.isfile(itr_file) else None)

!python -u ctg_approx/avi.py --env cube4 --back_max {BACK_MAX} --nnet_name cube4 --save_dir {SAVE_DIR} \
    --max_itrs {MAX_ITRS} --max_time {MAX_TIME_SEC} --batch_size {BATCH_SIZE} \
    --states_per_update {STATES_PER_UPDATE} --num_test {NUM_TEST} --update_nnet_batch_size 10000 \
    | grep -E "device:|Training model|Itr: [0-9]*00,|Back Steps|Last loss|Updating target|Time budget|Done"

print("checkpoint itr after:", pickle.load(open(itr_file, "rb")))

In [ ]:
# Evaluate (skips sections already in EVAL_JSON for this checkpoint)
!python scripts/step3_eval.py --model_dir {MODEL_DIR} --out {EVAL_JSON} {EVAL_ARGS}

In [ ]:
import json
from scripts.step3_eval import print_table
results = json.load(open(EVAL_JSON))
print("checkpoint itr:", results["config"]["model_itr"])
print_table(results)